In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import f1_score, accuracy_score
import wandb
from tqdm import tqdm


# Import from local files
from helper_functions import (
    seed_everything, generate_synthetic_dataset,
    extract_and_save_features, STEMS_PATH, NOISE_PATH, OUTPUT_PATH
)
from model_skeleton import CRNN, PrecomputedFeatureDataset

In [2]:
# --- Configuration ---
WORKING_BASE = '../working'
FEATURES_DIR = os.path.join(WORKING_BASE, 'features/train')
MODEL_DIR = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
# Step 1: Generate synthetic mashups (skip if already generated)
if not os.path.exists(OUTPUT_PATH) or len(os.listdir(OUTPUT_PATH)) == 0:
    print("Generating synthetic dataset...")
    seed_everything(42)
    generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)
else:
    print(f"Synthetic dataset already exists at {OUTPUT_PATH}, skipping generation.")

Synthetic dataset already exists at ../working\synthetic_mashups/train, skipping generation.


In [5]:
# Verify file count (Q1)
import glob
wav_count = len(glob.glob(os.path.join(OUTPUT_PATH, '**', '*.wav'), recursive=True))
print(f"Total .wav files generated: {wav_count}")

Total .wav files generated: 500


In [6]:
# Verify waveform shape (Q2)
import torchaudio
sample_wav = glob.glob(os.path.join(OUTPUT_PATH, '**', '*.wav'), recursive=True)[0]
waveform, sr = torchaudio.load(sample_wav)
print(f"Sample waveform shape: {waveform.shape}")

Sample waveform shape: torch.Size([2, 661500])


In [7]:
# Step 2: Extract features (skip if already done)
if not os.path.exists(FEATURES_DIR) or len(os.listdir(FEATURES_DIR)) == 0:
    print("Extracting mel-spectrogram features...")
    extract_and_save_features(OUTPUT_PATH, FEATURES_DIR)
else:
    print(f"Features already exist at {FEATURES_DIR}, skipping extraction.")

Features already exist at ../working\features/train, skipping extraction.


In [8]:
# Verify feature shape (Q3)
sample_pt = glob.glob(os.path.join(FEATURES_DIR, '**', '*.pt'), recursive=True)[0]
feature = torch.load(sample_pt)
print(f"Sample feature shape: {feature.shape}")


Sample feature shape: torch.Size([2, 128, 1292])


C:\Users\Atul\AppData\Local\Temp\ipykernel_27740\2131071811.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  feature = torch.load(sample_pt)


In [9]:
# Step 3: Create datasets and dataloaders
full_dataset = PrecomputedFeatureDataset(FEATURES_DIR)
print(f"Total samples: {len(full_dataset)}")

# 80/20 train-val split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size],
                                           generator=torch.Generator().manual_seed(42))

Total samples: 500


In [10]:
# Step 4: Build model and verify parameter counts
model = CRNN(num_classes=10).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [11]:
# Verify shape after second MaxPool2d (Q4)
sample_batch, _ = next(iter(train_loader))
sample_batch = sample_batch.to(DEVICE)
with torch.no_grad():
    cnn_out = model.cnn(sample_batch)
    print(f"Shape after CNN (second MaxPool2d): {cnn_out.shape}")

# LSTM trainable parameters (Q5)
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
print(f"LSTM trainable parameters: {lstm_params}")

d:\DLGenAI-project-Jan-2026\dl-genai-project-25-t3\notebooks\model_skeleton.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  feature = torch.load(file_path)


Shape after CNN (second MaxPool2d): torch.Size([32, 64, 32, 323])
LSTM trainable parameters: 1082368


In [12]:
# Step 5: Training loop with WandB
num_epochs = 10

wandb.init(
    project="21f3002298-dl-genai-project",
    name="crnn-milestone4",
    config={
        "model": "CRNN",
        "epochs": num_epochs,
        "batch_size": 32,
        "lr": 0.001,
        "optimizer": "Adam",
        "cnn_blocks": 2,
        "lstm_hidden": 64,
        "lstm_layers": 1,
        "bidirectional": True,
    }
)

best_val_f1 = 0.0

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Atul\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [13]:
for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []

    for features, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        features, labels = features.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * features.size(0)
        preds = outputs.argmax(dim=1).cpu().numpy()
        train_preds.extend(preds)
        train_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_dataset)
    train_acc = accuracy_score(train_labels, train_preds)
    train_f1 = f1_score(train_labels, train_preds, average='macro')

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for features, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            features, labels = features.to(DEVICE), labels.to(DEVICE)
            outputs = model(features)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * features.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_dataset)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    # Log to WandB
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
    })

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
          f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}")

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'crnn_best.pth'))
        print(f"  Saved best model (val_f1={val_f1:.4f})")

Epoch 1/10 [Train]:   0%|          | 0/13 [00:00<?, ?it/s]d:\DLGenAI-project-Jan-2026\dl-genai-project-25-t3\notebooks\model_skeleton.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experiment

Epoch 1: train_loss=2.1942 train_f1=0.2114 | val_loss=2.1146 val_f1=0.2314
  Saved best model (val_f1=0.2314)


Epoch 2/10 [Train]:   0%|          | 0/13 [00:00<?, ?it/s]d:\DLGenAI-project-Jan-2026\dl-genai-project-25-t3\notebooks\model_skeleton.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experiment

Epoch 2: train_loss=1.9286 train_f1=0.2813 | val_loss=1.8987 val_f1=0.2919
  Saved best model (val_f1=0.2919)


Epoch 3/10 [Train]:   0%|          | 0/13 [00:00<?, ?it/s]d:\DLGenAI-project-Jan-2026\dl-genai-project-25-t3\notebooks\model_skeleton.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experiment

Epoch 3: train_loss=1.7956 train_f1=0.3721 | val_loss=1.7941 val_f1=0.3493
  Saved best model (val_f1=0.3493)


... (10 outputs left out)

Epoch 9/10 [Train]:   0%|          | 0/13 [00:00<?, ?it/s]d:\DLGenAI-project-Jan-2026\dl-genai-project-25-t3\notebooks\model_skeleton.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experiment

Epoch 9: train_loss=1.1442 train_f1=0.7587 | val_loss=1.3321 val_f1=0.5462
  Saved best model (val_f1=0.5462)


Epoch 10/10 [Train]:   0%|          | 0/13 [00:00<?, ?it/s]d:\DLGenAI-project-Jan-2026\dl-genai-project-25-t3\notebooks\model_skeleton.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimen

Epoch 10: train_loss=1.0964 train_f1=0.7757 | val_loss=1.3573 val_f1=0.4978


In [14]:
wandb.finish()
print(f"\nTraining complete. Best val F1: {best_val_f1:.4f}")

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▄▄▅▆▆███
train_f1,▁▂▃▄▄▆▆███
train_loss,█▆▅▅▄▃▃▂▁▁
val_acc,▁▃▄▅▅▆▇▇█▇
val_f1,▁▂▄▄▅▆▇▇█▇
val_loss,█▆▅▅▄▃▂▂▁▁
epoch,10
train_acc,0.785
train_f1,0.77573
train_loss,1.09643



Training complete. Best val F1: 0.5462
